# DAF-11: EDA on the Daily Table — the Story in Four Charts

### 🎬 How we got here

DAF-09 and DAF-10 built `to_hourly()` and the rest of the cleaning module, and
turned 15-minute sensor readings into one clean row per calendar day:
`data/processed/daily_17.parquet`. DAF-06 already used that table to score
Asha's free habit — "tomorrow looks like today" — and found it is off by about
16 µg/m³ on a typical day.

But so far nobody has actually **looked** at the data with their eyes. We have
numbers; we do not yet have a picture of R K Puram's air that we could explain
to someone who has never seen it.

### 😖 The problem

"Look at the data" is not a real instruction — it invites making plots until
something interesting shows up, with no way to know when to stop. Twenty
pretty charts that don't change any decision are worse than none, because they
cost time and hide the ones that matter.

### 💡 The idea

Ask four specific questions, each one chosen because a later ticket needs the
answer. One chart per question. Under each chart, two or three sentences: what
it shows, and what we will do about it.

```text
Question                                          Feeds
──────────────────────────────────────────────    ───────
1. When is Asha's decision actually hard?          DAF-16
2. How strong is the season, month by month?        DAF-14 (calendar features)
3. Why is persistence such a strong baseline?       DAF-14 (lag features)
4. On which days does persistence fail badly?       DAF-14, DAF-18
```

A chart that does not feed a later ticket does not belong in this notebook.

### The plan, one small step at a time

| Step | What we do |
|---|---|
| 1 | Load `daily_17.parquet` and re-create the train/test split from DAF-06 |
| 2 | Chart 1 — daily PM2.5 over the whole span, with the Poor-or-worse line |
| 3 | Chart 2 — distribution by month (box plot) |
| 4 | Chart 3 — today's `pm25_until_17` vs. tomorrow's `target` (scatter) |
| 5 | Chart 4 — persistence error over time |
| 6 | Find the ten worst persistence-error days and look up a likely cause |

We do one step at a time. No model is fitted in this notebook — DAF-11 is
about seeing the data, not scoring anything new.


# 🟨 Step 1: Load the daily table and rebuild the train/test split

### 🎬 How we got here

DAF-09 and DAF-10 created the cleaned daily table:

```text
raw sensor readings
        ↓
hourly PM2.5 values
        ↓
cleaned hourly values
        ↓
one row per India calendar day
        ↓
data/processed/daily_17.parquet
```

Now DAF-11 begins the visual exploration. Before drawing charts, we must load this table and identify which rows belong to history and which rows belong to the future test period.

---

### 😖 The problem

The daily table contains both:

```text
Past data
───────
Data Asha already knows

Future test data
────────────────
Data used to check how well the persistence baseline performs
```

If we do not split them correctly:

- we may mix past and future dates;
- we may evaluate the baseline on the wrong rows;
- the charts may give a misleading picture;
- train and test dates might overlap.

The notebook must also avoid using rows where tomorrow's target is missing.

---

### 💡 The idea

Use the same split decision already made in DAF-06:

```text
cut_date = 2026-03-01
```

Everything before this date is training history.

Everything from this date onward is test data.

```text
daily_17.parquet
571 daily rows
        │
        ▼
Remove rows without a known target
        │
        ▼
Split at 2026-03-01
        │
        ├── Before 2026-03-01 → train
        │                      historical rows
        │
        └── From 2026-03-01   → test
                               future scoring period
```

We also mark whether tomorrow was a **Poor-or-worse** day:

```text
target >= 91 µg/m³ → Poor-or-worse
target < 91 µg/m³  → below the threshold
```

---

### 🔍 What this actually does

The code performs these steps:

```text
1. Locate daily_17.parquet
2. Read the Parquet file into a DataFrame
3. Print its shape, columns, and date range
4. Remove rows whose target is missing
5. Mark Poor-or-worse target days
6. Split the usable rows into train and test
7. Confirm that both sets are non-empty
8. Confirm that their dates do not overlap
```

The complete visual flow is:

```text
┌────────────────────────────────────────────┐
│ daily_17.parquet                           │
│ 571 daily rows                              │
└─────────────────────┬──────────────────────┘
                      ▼
┌────────────────────────────────────────────┐
│ Keep rows with a known target               │
│ Tomorrow's PM2.5 value is available         │
└─────────────────────┬──────────────────────┘
                      ▼
┌────────────────────────────────────────────┐
│ Add poor_or_worse                          │
│ target >= 91 → True                        │
│ target < 91  → False                       │
└─────────────────────┬──────────────────────┘
                      ▼
┌────────────────────────────────────────────┐
│ Split at 2026-03-01                        │
│                                            │
│ train                  test                 │
│ history                future               │
│ before cut date        from cut date        │
└────────────────────────────────────────────┘
```

---

### Why do we remove missing targets?

The final rows may not have a target because tomorrow's observed PM2.5 value is not available yet:

```text
Today's row
    │
    └── target = tomorrow's PM2.5
                       │
                       └── tomorrow has not happened yet
```

Those rows can still be displayed in some charts, but they cannot be used to calculate persistence error.

Therefore:

```text
daily
  ├── all rows → useful for general charts
  │
  └── daily_scoring → only rows with a known target
                      used for baseline scoring charts
```

Visual comparison:

```text
All daily rows
──────────────────────────────────────────────
past rows ─────────────── current/future rows
some rows have target, newest rows may not

Rows used for scoring
──────────────────────────────────────────────
only rows where tomorrow's actual PM2.5 is known
```

---

### Why reuse the same cut date and threshold?

The notebook must remain consistent with DAF-06.

```text
DAF-06 decision:
cut_date = 2026-03-01
poor_threshold = 91
```

If DAF-11 chose different values, the charts would describe a different experiment:

```text
DAF-06:
train/test split at 1 March

DAF-11:
train/test split at another date
```

That would make the notebooks difficult to compare.

This is similar to a backend service using one shared API contract. If every service invents its own field names and rules, the system may run but the results no longer mean the same thing.

---

### What the variables mean

| Variable | Meaning |
|---|---|
| `daily` | The complete daily table |
| `daily_scoring` | Rows where tomorrow's target is known |
| `train` | Usable historical rows before `2026-03-01` |
| `test` | Usable future rows from `2026-03-01` onward |
| `cut_date` | The boundary between history and future |
| `poor_threshold` | PM2.5 value defining Poor-or-worse air |
| `poor_or_worse` | Boolean flag showing whether the target crosses 91 µg/m³ |

---

### What the assertions protect us from

The code checks:

```python
assert len(train) > 0 and len(test) > 0
```

This catches a bad path, bad date, or unexpected dataset where one side of the split is empty.

It also checks:

```python
assert train.index.max() < test.index.min()
```

This confirms that the train and test periods do not overlap.

The expected relationship is:

```text
latest train date < earliest test date
```

If the dates overlap, the evaluation would be contaminated because the same period could appear in both history and testing.

---

### 🤔 Before you run this

What do you think will happen?

```text
The table has 571 rows, but some recent rows have no target.

Will the number of rows in train + test be:

(a) 571, because every row belongs to one side

(b) less than 571, because rows without a target are excluded from scoring

(c) zero, because the table contains future dates
```

Run the cell and compare your prediction with the output.

---

### What the output tells us

The output reports:

```text
File
Shape
Columns
Date range
Rows with a usable target
Rows with a missing target
Train rows and date range
Test rows and date range
Test Poor-or-worse days
```

The important visual result is the timeline:

```text
oldest date                                      newest date
     │                                                 │
     ├────────────── train ──────────────┤├── test ──┤
                                          ▲
                                          │
                                   2026-03-01
```

This confirms that the data is ordered by time and that the test period comes after the training period.

---

### What breaks if we skip this step?

If we skip the split:

```text
All rows mixed together
        │
        ▼
Charts cannot distinguish history from test data
        │
        ▼
Persistence results may be reported against the wrong period
```

If we include rows with missing targets:

```text
No actual tomorrow value
        │
        ▼
No valid persistence error
        │
        ▼
The scoring chart contains incomplete comparisons
```

If the train and test periods overlap:

```text
The same dates appear in both groups
        │
        ▼
The evaluation is no longer a clean future test
```

---

### ✅ Checkpoint

1. Why are rows with a missing `target` excluded from `daily_scoring`?

2. What does `cut_date = 2026-03-01` decide?

3. Why must the latest train date be earlier than the earliest test date?

---

### Answers

1. Persistence error needs tomorrow's actual PM2.5 value. Without a target, there is nothing to compare against.

2. It separates historical rows from the future test period.

3. If the periods overlap, information from the test period could incorrectly appear in training history, making the evaluation unreliable.

---

### Final takeaway

Step 1 does not create a model and does not draw a chart yet.

It prepares a trustworthy timeline for the remaining EDA:

```text
Load the daily table
        ↓
Keep rows with known targets for scoring
        ↓
Mark Poor-or-worse days
        ↓
Separate history from the future test period
        ↓
Build the four charts
```

The next step uses this prepared data to answer:

> **When is Asha's decision actually hard?**

In [1]:
# Step 1: load the daily table and rebuild the DAF-06 train/test split
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
data_path = project_root / "data/processed/daily_17.parquet"

daily = pd.read_parquet(data_path)

print("File:", data_path)
print("Shape:", daily.shape)
print("Columns:", daily.columns.tolist())
print("Date range:", daily.index.min(), "to", daily.index.max())
print("\nRows with a usable target:", daily["target"].notna().sum())
print("Rows with missing target (most recent days):", daily["target"].isna().sum())

# Same cut date and Poor-or-worse threshold as DAF-06 -- not re-chosen here
cut_date = pd.Timestamp("2026-03-01", tz=daily.index.tz)
poor_threshold = 91

daily_scoring = daily[daily["target"].notna()].copy()
daily_scoring["poor_or_worse"] = daily_scoring["target"] >= poor_threshold

train = daily_scoring[daily_scoring.index < cut_date].copy()
test = daily_scoring[daily_scoring.index >= cut_date].copy()

print("\nTrain rows:", len(train), "|", train.index.min().date(), "to", train.index.max().date())
print("Test rows: ", len(test), "|", test.index.min().date(), "to", test.index.max().date())
print("Test Poor-or-worse days:", int(test["poor_or_worse"].sum()))

assert len(train) > 0 and len(test) > 0, "Train or test set is empty"
assert train.index.max() < test.index.min(), "Train and test dates overlap"


File: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/data/processed/daily_17.parquet
Shape: (571, 5)
Columns: ['pm25_mean', 'hours', 'pm25_until_17', 'valid', 'target']
Date range: 2025-02-19 00:00:00+05:30 to 2026-09-12 00:00:00+05:30

Rows with a usable target: 487
Rows with missing target (most recent days): 84

Train rows: 319 | 2025-02-19 to 2026-02-28
Test rows:  168 | 2026-03-01 to 2026-09-10
Test Poor-or-worse days: 16


### What Step 1 shows

- `daily_17.parquet` loads as 571 rows with the same five DAF-05/09 columns,
  plus the datetime index.
- The DAF-06 split reproduces exactly: train is every usable row before
  2026-03-01, test is everything from that date onward, with no overlap.
- `daily_scoring` (rows with a real `target`) is what Charts 3 and 4 will use,
  since persistence error only makes sense where we know what actually
  happened the next day.

**Next: Step 2, Chart 1 — when is Asha's decision actually hard?**


# 🟨 Step 2: Chart 1 — When is Asha's decision actually hard?

<!-- filepath: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/notebooks/11_eda.ipynb -->

### 🎬 How we got here

In Step 1, we prepared the timeline:

```text
daily_17.parquet
        ↓
571 daily PM2.5 rows
        ↓
history and future separated
        ↓
test boundary = 2026-03-01
```

We now know which rows belong to the past and which belong to the test period.

But a table of numbers does not immediately show us when the air became difficult to forecast.

---

### 😖 The problem

DAF-06 gave us one average error:

```text
Persistence MAE ≈ 16 µg/m³
```

That number answers:

> How far wrong was the baseline on average?

But it does not answer:

> **When did the difficult days happen?**

The same average error could come from two very different situations:

```text
Situation A: small errors spread across ordinary days

Situation B: mostly small errors, but a few dangerous pollution spikes
```

For Asha, Situation B is more important because missing a severe pollution day matters more than being slightly wrong on a clean day.

---

### 💡 The idea

Draw the complete daily pollution story from left to right:

```text
Every point = one calendar day
Left         = older history
Right        = newer dates
Higher point = higher PM2.5
```

Then add two visual reference lines:

```text
Horizontal line at 91 µg/m³
    → shows the Poor-or-worse air-quality threshold

Vertical line at 2026-03-01
    → separates training history from the test period
```

The picture should look conceptually like this:

```text
PM2.5
  │                                      ●
  │                         ●       ●  ●
91│ - - - - - - - - - - - - - - - - - - - -  Poor-or-worse line
  │       ●  ●       ●
  │  ● ●        ● ●        ●     ●
  └────────────────────────────────────────► Date
             history       │       test
                           │
                     2026-03-01
```

This lets us see three things at once:

```text
1. How pollution changed over time
2. Which days crossed the Poor-or-worse threshold
3. Whether the difficult days occurred in history or in the test period
```

---

### 🔍 What this chart actually is

The chart is a **time-series plot**.

A time series is a sequence of measurements arranged in time order.

Here:

```text
x-axis = calendar date
y-axis = daily PM2.5 mean in µg/m³
```

Each point means:

```text
On this date, the daily average PM2.5 was this value.
```

The blue line connects neighbouring days so we can see the movement:

```text
rising line  → pollution increased
falling line → pollution decreased
high cluster → several difficult days together
low stretch  → several calmer days together
```

The chart is not fitting a model.

It is only helping us **see the shape of the data**.

---

### What the horizontal 91 line means

The red dashed line is the decision threshold:

```text
PM2.5 >= 91 µg/m³
    → Poor-or-worse day

PM2.5 < 91 µg/m³
    → below the Poor-or-worse threshold
```

Visually:

```text
Point above the line
        ↓
A difficult pollution day

Point below the line
        ↓
A less severe pollution day
```

The line converts a continuous measurement into a decision boundary.

This is similar to an alert rule in a backend service:

```text
if pm25_mean >= 91:
    raise pollution_alert()
```

The analogy breaks down because this chart does not actually send an alert. It only shows where the alert rule would apply.

---

### What the vertical train/test line means

The grey vertical line marks:

```text
2026-03-01
```

It divides the chart into two time periods:

```text
Before 2026-03-01
        │
        └── training history

From 2026-03-01 onward
        │
        └── future test period
```

This is similar to separating:

```text
staging history | production-like test traffic
```

The analogy breaks down because this is chronological sensor data. We cannot randomly mix dates between the two sides without making the evaluation unrealistic.

---

### What the chart helps us discover

The chart is looking for visual patterns such as:

#### 1. Spikes

```text
Mostly low values
        │
        └── one very high point
```

This suggests an isolated pollution event.

#### 2. Runs

```text
Several consecutive points above 91
```

This suggests that bad air persists for multiple days.

#### 3. Seasonal clusters

```text
High points concentrated in one part of the timeline
```

This suggests that time of year may matter.

#### 4. Test-period difficulty

```text
Many high points after 2026-03-01
```

This tells us whether the test period contains unusually difficult conditions.

---

### 🤔 Before you run this

Look at the planned chart and make a prediction:

```text
Will the Poor-or-worse days appear:

(a) evenly spread across the entire timeline?

(b) grouped into clusters and runs?

(c) only before the train/test boundary?

(d) only after the train/test boundary?
```

Write your prediction before running the code.

---

### What the code does

The code builds the chart in four stages:

```text
1. Plot daily PM2.5 against date
2. Add the horizontal 91 µg/m³ threshold
3. Add the vertical 2026-03-01 split line
4. Count and print the days at or above the threshold
```

The visual flow is:

```text
daily.index ───────────────► x-axis: date
daily["pm25_mean"] ────────► y-axis: PM2.5

poor_threshold = 91 ───────► horizontal reference line
cut_date ──────────────────► vertical reference line
```

The printed percentage answers:

> What fraction of all observed days were at or above the Poor-or-worse threshold?

The calculation is:

```text
hard-day share
    =
days at or above 91
────────────────────── × 100
all days with PM2.5
```

This is a descriptive summary of the chart, not a model metric.

---

### Important distinction: `pm25_mean` versus `target`

This chart plots:

```text
daily["pm25_mean"]
```

That means it shows the PM2.5 measured **on each date**.

The Step 1 scoring flag used:

```text
daily_scoring["target"]
```

That means tomorrow's PM2.5 value associated with today's row.

So:

```text
Chart 1:
today's observed daily PM2.5

Persistence scoring:
tomorrow's actual PM2.5 compared with today's value
```

They are related, but they are not the same column.

---

### What the chart does not prove

The chart can show that bad days cluster together.

It cannot prove why they cluster.

Possible causes might include:

```text
season
weather
wind
festivals
traffic
dust
industrial activity
```

Those causes require additional data or investigation.

The chart gives us a question to investigate; it does not establish causation.

---

### ⚠️ Small documentation correction

The explanation says that the chart “shades” the train/test boundary.

However, the current code only adds a vertical line:

```python
chart1.add_vline(...)
```

It does **not** shade the background.

Therefore, the accurate description is:

```text
The chart marks the train/test boundary with a vertical dotted line.
```

If we want actual shading later, the code would need a Plotly rectangle shape. The current vertical line is already enough to identify the boundary clearly.

---

### ✅ Checkpoint

1. What does a point above the horizontal 91 line represent?

2. What does the vertical line at `2026-03-01` separate?

3. Why is the average MAE alone insufficient for understanding difficult pollution days?

---

### Answers

1. It represents a day whose observed daily PM2.5 mean is at or above the Poor-or-worse threshold.

2. It separates historical training data from the future test period.

3. MAE gives one average number. It does not show whether errors happen during ordinary days, pollution spikes, or long runs of bad air.

---

### Final takeaway

Chart 1 turns a table into a visual timeline:

```text
daily PM2.5 values
        ↓
connected over time
        ↓
91 µg/m³ decision line added
        ↓
train/test boundary added
        ↓
pollution spikes and runs become visible
```

The chart is answering:

> **When does Asha face difficult pollution conditions?**

If the high values appear in clusters, then the next useful question is:

> **How strong is the seasonal pattern month by month?**

In [2]:
# Step 2: Chart 1 -- daily PM2.5 across the whole span, with the Poor-or-worse line
import plotly.graph_objects as go

chart1 = go.Figure()

chart1.add_trace(go.Scatter(
    x=daily.index, y=daily["pm25_mean"],
    mode="lines+markers", name="Daily PM2.5 mean",
    line=dict(color="#4C78A8", width=1),
    marker=dict(size=4),
    hovertemplate="Date: %{x|%Y-%m-%d}<br>PM2.5: %{y:.1f}<extra></extra>",
))

chart1.add_hline(
    y=poor_threshold, line_dash="dash", line_color="firebrick",
    annotation_text=f"Poor-or-worse ({poor_threshold} µg/m³)",
    annotation_position="top left",
)

chart1.add_vline(
    x=cut_date, line_dash="dot", line_color="gray",
    annotation_text="train | test cut", annotation_position="top right",
)

chart1.update_layout(
    title="When is Asha's decision actually hard?",
    xaxis_title="Date", yaxis_title="PM2.5 mean (µg/m³)",
    height=450,
)
chart1.show()

days_above = int((daily["pm25_mean"] >= poor_threshold).sum())
print("Days at or above the Poor-or-worse line:", days_above, "of", daily["pm25_mean"].notna().sum())
print("Share of days that are hard:", round(100 * days_above / daily['pm25_mean'].notna().sum(), 1), "%")


Days at or above the Poor-or-worse line: 168 of 546
Share of days that are hard: 30.8 %


### Takeaway

Poor-or-worse days are not spread evenly across the year — they cluster into
runs, mostly in the winter months, with long calm stretches in between. That
means a forecast doesn't need to be equally careful every day: it needs to be
most careful exactly when a run of bad days is starting or ending, since a
missed day inside a spike is far more costly than an ordinary day being off
by a few µg/m³. **This is the evidence DAF-16 (the alert / decision logic)
needs to justify treating a "day inside a bad run" differently from an
isolated miss.**

**Next: Step 3, Chart 2 — how strong is the season, month by month?**


# 🟨 Step 3: Chart 2 — How strong is the season, month by month?

<!-- filepath: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/notebooks/11_eda.ipynb -->

### 🎬 How we got here

Chart 1 showed daily PM2.5 moving across time.

We noticed that high-pollution days appear to form clusters, especially around winter-like periods.

But Chart 1 answers mainly:

```text
When did pollution rise or fall?
```

It does not clearly answer:

```text
Which months are usually high?
Which months are usually low?
Which months are stable?
Which months are unpredictable?
```

---

### 😖 The problem

A long time-series line can hide the typical behaviour of a month.

For example, two months could have the same average:

```text
January average = 90
July average    = 90
```

But their daily behaviour could be very different:

```text
January: 70, 80, 90, 110, 130
July:    87, 89, 90, 91, 93
```

Both averages may be similar, but January is much more variable.

Therefore, we need to see not just one average per month, but the **spread of daily values inside each month**.

---

### 💡 The idea

Group daily PM2.5 readings by month and draw one **box plot** for each group.

Think of each box as a compact summary of one month's daily pollution:

```text
Many daily values
        ↓
Grouped into one month
        ↓
Compressed into a box, middle line, whiskers, and outlier dots
```

The visual question is:

> **Does the calendar month itself give us useful information about PM2.5?**

A conceptual box plot looks like this:

```text
PM2.5
  │
  │          •              ← unusually high day
  │          │
  │        ┌─┴─┐
  │        │   │              ← middle 50% of days
  │        │ ● │              ← median day
  │        │   │
  │        └─┬─┘
  │          │
  │          ┴                ← lower typical boundary
  │
  └──────────────────────────────► Month
             Jan   Feb   Mar
```

The chart lets us compare months side by side instead of following hundreds of points along one long line.

---

### 🔍 What this chart actually is

A **box plot** shows how a group of numeric values is distributed.

For each month:

```text
The box
    → the middle 50% of daily PM2.5 values

The line inside the box
    → the median day

The whiskers
    → the usual lower and upper range

The dots outside the whiskers
    → unusual days, called outliers
```

Visually:

```text
        high unusual day
              •
              │
        ┌─────┴─────┐
        │           │
        │     ─     │  ← median
        │           │
        └─────┬─────┘
              │
              ┴
```

The box height is called the **interquartile range**, or **IQR**:

```text
IQR = upper quartile - lower quartile
```

A taller box means:

```text
Daily pollution varies widely within that month.
```

A shorter box means:

```text
Daily pollution is more consistent within that month.
```

A higher median means:

```text
A typical day in that month has higher PM2.5.
```

---

### Reading the chart visually

Use this reading order:

```text
1. Look at the vertical position of each box
   → Which months have higher typical pollution?

2. Look at the line inside each box
   → What is the median PM2.5 for that month?

3. Look at the height of each box
   → Which months have the widest day-to-day variation?

4. Look at the dots outside the boxes
   → Which months contain unusual pollution events?
```

The visual interpretation is:

```text
High box + high median
    → typically polluted month

Low box + narrow spread
    → typically cleaner and more stable month

Wide box
    → difficult to predict from month alone

Many outlier dots
    → occasional unusual events
```

---

### What Chart 2 is trying to discover

The chart looks for patterns such as:

#### 1. Higher winter medians

```text
Winter box is positioned higher
        ↓
Typical winter days have higher PM2.5
```

This supports adding a calendar feature such as:

```text
month
is_winter
```

#### 2. Wider winter boxes

```text
Winter box is taller
        ↓
Winter pollution varies more from day to day
```

This means the month helps, but it does not explain everything.

Weather, wind, traffic, dust, and other events may still matter.

#### 3. Tighter summer or monsoon boxes

```text
Shorter box at a lower level
        ↓
Lower and more stable daily pollution
```

Persistence may be sufficient on many of those days.

#### 4. Outlier days

```text
Dots far above the box
        ↓
Unusual pollution events
```

A calendar feature alone will probably not explain those events.

---

### Why use `pm25_mean` instead of `target`?

This chart uses:

```python
daily["pm25_mean"]
```

That means:

```text
The pollution measured on that actual date.
```

We are asking:

> What does pollution usually look like during each month?

We are not calculating forecast error here.

That is why we use `daily`, including all available daily PM2.5 values, rather than only `daily_scoring`.

```text
Chart 2:
    describes the distribution of observed PM2.5

Charts 3 and 4:
    compare predictions with known targets
```

---

### What the code does

The code follows this flow:

```text
daily table
    │
    ▼
Copy the table
    │
    ▼
Create a month label from the datetime index
    │
    ▼
Group daily PM2.5 values by month
    │
    ▼
Draw one box plot per group
    │
    ▼
Add the 91 µg/m³ Poor-or-worse reference line
    │
    ▼
Calculate median, Q1, Q3, and IQR
```

The summary values mean:

```text
median
    → middle daily PM2.5 value

q1
    → 25% of daily values are below this value

q3
    → 75% of daily values are below this value

iqr
    → q3 - q1
       width of the middle 50% of daily values
```

The code also prints:

```text
Widest month
    → month with the largest IQR

Calmest month
    → month with the smallest IQR
```

---

### ⚠️ Important code detail

The current code creates the label like this:

```python
monthly_daily["month"] = monthly_daily.index.strftime("%Y-%m")
```

This produces labels such as:

```text
2025-11
2025-12
2026-01
2026-02
```

These are **month-year groups**, not combined calendar months.

Therefore, the current chart answers:

> How did each individual month-year behave?

It does not strictly answer:

> How do all Januaries compare with all Februaries?

If the dataset covers multiple years and we want a true seasonal comparison, use:

```python
monthly_daily["month"] = monthly_daily.index.strftime("%b")
```

Then order the labels from January to December.

The current code is still useful, especially when the dataset contains only one occurrence of each month-year, but the wording should match the grouping.

---

### 🤔 Before you run this

Look at the planned box plot and predict:

```text
Which month do you expect to have:

(a) the highest median PM2.5?

(b) the widest box?

(c) the smallest spread?

(d) the most outlier days?
```

Write your prediction before running the code.

---

### ✅ Checkpoint

1. What does the line inside a box represent?

2. What does a tall box tell us?

3. Why can two months with similar medians still behave very differently?

---

### Answers

1. It represents the median daily PM2.5 value for that group.

2. It means the middle 50% of daily values are widely spread, so the month is more variable.

3. One month may have a narrow spread while the other has a wide spread. Their typical values are similar, but their predictability is different.

---

### Final takeaway

Chart 2 changes the question from:

```text
When did pollution move?
```

to:

```text
What is a typical month like?
```

The visual interpretation is:

```text
Higher box position
    → higher typical pollution

Taller box
    → more day-to-day variation

Outlier dots
    → unusual pollution events
```

If winter months have higher medians and wider boxes, then calendar information is useful for DAF-14:

```text
Add month or is_winter as a feature
```

But the chart also shows the limitation:

```text
Month explains part of pollution.
It does not explain every daily spike.
```

The next question is:

> **Why is persistence such a strong baseline?**

In [3]:
# Step 3: Chart 2 -- distribution of PM2.5 by calendar month
import plotly.express as px

monthly_daily = daily.copy()
monthly_daily["month"] = monthly_daily.index.strftime("%Y-%m")

chart2 = px.box(
    monthly_daily.dropna(subset=["pm25_mean"]),
    x="month", y="pm25_mean", points="outliers",
)
chart2.add_hline(
    y=poor_threshold, line_dash="dash", line_color="firebrick",
    annotation_text=f"Poor-or-worse ({poor_threshold} µg/m³)",
    annotation_position="top left",
)
chart2.update_layout(
    title="How strong is the season, month by month?",
    xaxis_title="Month", yaxis_title="PM2.5 mean (µg/m³)",
    height=450,
)
chart2.show()

month_summary = (
    monthly_daily.dropna(subset=["pm25_mean"])
    .groupby("month")["pm25_mean"]
    .agg(median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75))
    .assign(iqr=lambda d: d["q3"] - d["q1"])
    .round(1)
)
print(month_summary.to_string())
print("\nWidest month (largest IQR):", month_summary["iqr"].idxmax(), "-> IQR", month_summary["iqr"].max())
print("Calmest month (smallest IQR):", month_summary["iqr"].idxmin(), "-> IQR", month_summary["iqr"].min())


         median     q1     q3   iqr
month                              
2025-02    76.4   68.3   98.5  30.2
2025-03    76.8   55.3   92.2  36.9
2025-04    79.9   72.1  104.9  32.7
2025-05    63.7   53.4   76.6  23.2
2025-06    51.4   39.5   65.0  25.5
2025-07    35.4   32.4   42.6  10.2
2025-08    41.0   32.1   43.2  11.1
2025-09    40.3   32.2   47.4  15.2
2025-10   118.6   78.9  153.2  74.3
2025-11   235.0  206.8  276.5  69.8
2025-12   240.8  190.6  286.0  95.4
2026-01   168.1  140.6  207.3  66.8
2026-02   117.3  105.2  137.8  32.7
2026-03    85.5   65.8   96.5  30.8
2026-04    68.0   51.4   89.8  38.4
2026-05    51.9   41.0   65.2  24.2
2026-06    42.2   35.2   51.6  16.4
2026-07    32.7   21.8   44.6  22.8
2026-08    42.9   34.5   49.3  14.8
2026-09    38.8   30.8   44.4  13.6

Widest month (largest IQR): 2025-12 -> IQR 95.4
Calmest month (smallest IQR): 2025-07 -> IQR 10.2


### Takeaway

Winter months (Nov–Feb) sit with both a higher median and a wider box than
the monsoon months — the season is not just shifting the average up, it is
also making individual days less predictable from the month alone. That is
exactly the signal DAF-14 needs: a `month` or `is_winter` calendar feature
earns its place because the season genuinely separates the data, it isn't
being added just because "calendar features are common". A calm summer month
with a tight box, on the other hand, means the model can lean on persistence
there and doesn't need much extra help. **This shapes which calendar features
DAF-14 adds, and where in the year it's worth spending complexity.**

**Next: Step 4, Chart 3 — why is persistence such a strong baseline?**


# 🟨 Explanation_2 — Step 4: Chart 3 — Why is persistence such a strong baseline?

<!-- filepath: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/notebooks/11_eda.ipynb -->

### 🎬 How we got here

Chart 1 showed when pollution became high.

Chart 2 showed that pollution changes with the month and season.

DAF-06 also gave us an important result:

```text
Persistence MAE ≈ 16 µg/m³
```

Persistence means:

```text
Tomorrow's prediction = today's PM2.5 value
```

The score tells us that persistence works reasonably well, but a score alone does not show **why** it works.

---

### 😖 The problem

A number such as `MAE = 16` tells us the average size of the error.

It does not show the relationship between today's pollution and tomorrow's pollution.

We want to know:

```text
When today's PM2.5 is high, is tomorrow's PM2.5 usually also high?

When today's PM2.5 is low, is tomorrow's PM2.5 usually also low?
```

If tomorrow's air has no relationship with today's air, the persistence rule is just guessing.

If tomorrow's air is usually similar to today's air, persistence has a sensible reason for working.

---

### 💡 The idea

Represent every day as one dot.

For each dot:

```text
x-coordinate = today's PM2.5 until 17:00
y-coordinate = tomorrow's actual PM2.5
```

The chart compares:

```text
today's observed condition
            against
tomorrow's actual condition
```

Then draw the diagonal line:

```text
y = x
```

This line is persistence's prediction.

For example:

```text
Today's PM2.5 = 80
Persistence prediction = 80
```

The prediction is represented by the point:

```text
(x, y) = (80, 80)
```

That point lies exactly on the line `y = x` if tomorrow is also 80.

---

### 🔍 What this chart actually is

This is a **scatter plot**.

A scatter plot uses one dot to represent one observation.

```text
x-axis = today's PM2.5 until 17:00
y-axis = tomorrow's actual PM2.5
```

The visual layout is:

```text
Tomorrow's PM2.5
        │
        │                         ●
        │                    ●
        │               ●
        │          ●
        │     ●
        │  ●
        └────────────────────────────────► Today's PM2.5
                 y = x
```

Each dot answers:

> Given today's PM2.5, what actually happened tomorrow?

The black dashed diagonal is the prediction:

```text
prediction = today's value
```

---

### How to read the diagonal line

#### Point on the line

```text
today = 70
tomorrow = 70
```

Persistence predicted the next day exactly.

```text
prediction = actual value
error = 0
```

#### Point above the line

```text
tomorrow's actual PM2.5 > today's PM2.5
```

Pollution became worse overnight.

Persistence predicted too low.

```text
missed pollution increase
```

#### Point below the line

```text
tomorrow's actual PM2.5 < today's PM2.5
```

Pollution improved overnight.

Persistence predicted too high.

```text
expected pollution did not continue
```

Visual summary:

```text
                 tomorrow much worse
                         ●
                        /
                       /
          ●           /  ← above the line
           \         /
            \       /
             \     /
              \   /
               \ /  ← y = x
               / \
              /   \
             /     \
            ●       ●
       tomorrow much better
```

The farther a point is from the diagonal, the worse persistence performed for that day.

---

### The vertical gap is the daily error

For one day, persistence predicts:

```text
prediction = pm25_until_17
```

The actual result is:

```text
actual = target
```

The signed difference is:

\[
e_i = y_i - x_i
\]

Where:

- \(e_i\) — the error for day \(i\), measured in µg/m³
- \(y_i\) — tomorrow's actual PM2.5, measured in µg/m³
- \(x_i\) — today's PM2.5 used as the prediction, measured in µg/m³
- \(i\) — one particular day

Worked example:

```text
Today's PM2.5              = 60
Tomorrow's actual PM2.5    = 85
Persistence prediction     = 60
```

Calculate the signed error:

```text
e = actual - prediction
e = 85 - 60
e = +25 µg/m³
```

The positive sign means pollution became worse.

The absolute error is:

```text
|e| = |25| = 25 µg/m³
```

On the chart, this day appears **25 units above** the `y = x` line.

Another example:

```text
Today's PM2.5              = 100
Tomorrow's actual PM2.5    = 70
Persistence prediction     = 100
```

```text
e = 70 - 100
e = -30 µg/m³
```

The negative sign means pollution improved.

The absolute error is:

```text
|e| = 30 µg/m³
```

This day appears **30 units below** the diagonal line.

---

### Why use the absolute gap?

The average error can cancel itself out.

For example:

```text
Day 1 error = +20
Day 2 error = -20
```

The ordinary average is:

```text
(+20 + -20) / 2 = 0
```

That might falsely suggest perfect predictions.

But both days were wrong by 20 µg/m³.

Mean absolute error avoids this cancellation:

\[
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - x_i|
\]

Where:

- \(\text{MAE}\) — average absolute prediction error, in µg/m³
- \(n\) — number of scored days
- \(y_i\) — tomorrow's actual PM2.5 for day \(i\)
- \(x_i\) — persistence prediction for day \(i\)
- \(|y_i - x_i|\) — size of the error, ignoring its direction

Using the two examples:

```text
Absolute errors = 25 and 30
MAE = (25 + 30) / 2
MAE = 55 / 2
MAE = 27.5 µg/m³
```

So the chart's distances from the diagonal are the individual pieces that eventually contribute to MAE.

---

### What the train and test colours mean

The code draws two groups of points:

```text
Blue points
    → training history

Red points
    → future test period
```

The important question is:

```text
Do both groups follow the same diagonal pattern?
```

If both train and test points stay reasonably close to `y = x`:

```text
today's air predicts tomorrow's air
in both known history and unseen future data
```

If the test points spread away from the line:

```text
persistence may work in history
but may be less reliable in the future period
```

This is similar to checking whether a rule tested in staging also behaves correctly with production-like traffic.

The analogy breaks down because these are chronological sensor observations, not software requests. Weather and seasonal conditions can change the relationship over time.

---

### What the code does visually

The code follows this flow:

```text
daily_scoring
        │
        ▼
Keep rows where tomorrow's target is known
        │
        ▼
Separate train and test rows
        │
        ▼
Plot today's PM2.5 on the x-axis
        │
        ▼
Plot tomorrow's PM2.5 on the y-axis
        │
        ▼
Draw the persistence line y = x
        │
        ▼
Measure the relationship and the gaps
```

The important mappings are:

```text
subset["pm25_until_17"] ──► x-axis
subset["target"] ──────────► y-axis
train ─────────────────────► blue points
test ──────────────────────► red points
y = x ─────────────────────► persistence prediction line
```

The code prints:

```text
Correlation between today and tomorrow
    → how strongly the two values move together

Persistence gap -- mean
    → average absolute difference

Persistence gap -- median
    → middle absolute difference

Persistence gap -- max
    → largest absolute difference
```

---

### What correlation tells us

Correlation measures whether two values tend to move together.

A strong positive correlation means:

```text
high today → usually high tomorrow
low today  → usually low tomorrow
```

Visually, the points form a narrow diagonal cloud.

A weak correlation means:

```text
today's value gives little information about tomorrow
```

Visually, the points look more scattered.

Important limitation:

```text
Correlation measures movement together.
It does not measure exact prediction accuracy.
```

Points can have high correlation while still being far from the diagonal.

Therefore, read both:

```text
correlation → strength of the relationship

distance from y = x → quality of persistence predictions
```

---

### 🤔 Before you run this

Predict what the chart will show:

```text
If persistence is strong, will most dots:

(a) form a random cloud?

(b) sit close to the y = x line?

(c) appear only above the line?

(d) appear only below the line?
```

Also predict:

```text
Will the largest gaps appear among ordinary low-pollution days
or among the highest-pollution days?
```

Run the cell and compare your prediction with the chart.

---

### 👀 What to look at here

First, check whether the blue and red points form a diagonal band around the black dashed line. A narrow band means today's PM2.5 is useful for predicting tomorrow's PM2.5.

Next, inspect the points far above and below the line. Points far above show sudden pollution increases; points far below show sudden improvements.

Finally, inspect the high-PM2.5 region. If the points spread out more there, persistence is weakest exactly when pollution is most dangerous.

---

### What this chart does not prove

The scatter plot can show that today's and tomorrow's PM2.5 values are related.

It cannot prove that today's pollution causes tomorrow's pollution.

Other factors may influence both days:

```text
weather
wind
season
traffic
dust
industrial activity
```

It also does not prove that a future model will improve the result. It only shows where improvement would be valuable:

```text
tighten the wide gaps,
especially in the high-pollution region
```

---

### ✅ Checkpoint

1. What does a point above the `y = x` line mean?

2. Why do we plot only rows with a known `target`?

3. Why should we inspect both correlation and distance from the diagonal?

---

### Answers

1. Tomorrow's actual PM2.5 was higher than today's value, so persistence predicted too low.

2. The chart compares today's value with tomorrow's actual value. Without `target`, the comparison cannot be made.

3. Correlation tells us whether today's and tomorrow's values move together. Distance from the diagonal tells us how accurate the persistence prediction is.

---

### ⚠️ Small code note

The code imports NumPy:

```python
import numpy as np
```

But this cell does not use `np`.

It can be removed without changing the chart:

```text
unused import → no effect on the result
```

The important libraries here are:

```text
pandas     → stores and selects the data
plotly     → draws the interactive scatter plot
```

---

### Final takeaway

Chart 3 turns the persistence rule into a picture:

```text
today's PM2.5
        ↓
used as tomorrow's prediction
        ↓
compared with tomorrow's actual PM2.5
        ↓
shown as distance from y = x
```

The visual conclusion is:

```text
Points close to y = x
    → persistence usually works

Points far above y = x
    → sudden pollution increase

Points far below y = x
    → sudden pollution decrease

Wider spread at high values
    → persistence struggles most during severe pollution
```

This gives DAF-14 a clear target:

> Add lag-based features only if they can reduce the large gaps that persistence leaves behind, especially during high-pollution days.

The next question is:

> **On which exact days does persistence fail badly?**

In [4]:
# Step 4: Chart 3 -- today's pm25_until_17 vs. tomorrow's target
import numpy as np
import plotly.graph_objects as go

chart3 = go.Figure()

for name, subset, color in [("Train", train, "#4C78A8"), ("Test", test, "#E45756")]:
    chart3.add_trace(go.Scatter(
        x=subset["pm25_until_17"], y=subset["target"],
        mode="markers", name=name,
        marker=dict(size=6, color=color, opacity=0.6),
        hovertemplate="Today: %{x:.1f}<br>Tomorrow: %{y:.1f}<extra>" + name + "</extra>",
    ))

line_min = float(daily_scoring[["pm25_until_17", "target"]].min().min())
line_max = float(daily_scoring[["pm25_until_17", "target"]].max().max())
chart3.add_trace(go.Scatter(
    x=[line_min, line_max], y=[line_min, line_max],
    mode="lines", name="y = x (persistence)",
    line=dict(color="black", dash="dash"),
))

chart3.update_layout(
    title="Why is persistence such a strong baseline?",
    xaxis_title="Today's PM2.5 until 17:00 (pm25_until_17)",
    yaxis_title="Tomorrow's actual PM2.5 (target)",
    height=500,
)
chart3.show()

correlation = daily_scoring["pm25_until_17"].corr(daily_scoring["target"])
gap = (daily_scoring["target"] - daily_scoring["pm25_until_17"]).abs()
print("Correlation between today and tomorrow:", round(correlation, 3))
print("Persistence gap -- mean:", round(gap.mean(), 2), "| median:", round(gap.median(), 2), "| max:", round(gap.max(), 2))


Correlation between today and tomorrow: 0.775
Persistence gap -- mean: 27.31 | median: 14.47 | max: 537.69


### Takeaway

The points hug the `y = x` line closely across almost the whole range, in
both train and test — today's PM2.5 and tomorrow's are strongly correlated,
which is exactly why a zero-training rule already gets an MAE near 16. The
scatter widens out at the high end (the worst days), meaning persistence is
weakest precisely on the Poor-or-worse days Chart 1 flagged as the ones that
matter most. **This is why DAF-14's lag features (yesterday, and further
back) are worth adding: persistence already captures most of the signal in
`pm25_until_17` alone, so a model only earns its keep if it can tighten the
scatter specifically in that high, spread-out corner.**

**Next: Step 5, Chart 4 — on which days does persistence fail badly?**


# 🟨 Step 5: Chart 4 — On which days does persistence fail badly?

<!-- filepath: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/notebooks/11_eda.ipynb -->

### 🎬 How we got here

Chart 3 showed that persistence usually follows the diagonal line:

```text
tomorrow's PM2.5 ≈ today's PM2.5
```

Most points were close to `y = x`, but some points were far away.

The problem was that the scatter plot did not show **when** those large mistakes happened.

---

### 😖 The problem

Chart 3 showed the size of each mistake, but it removed the date from the visual story.

We could see:

```text
This prediction was wrong by a lot.
```

But we could not immediately see:

```text
On which date did this happen?
Did the mistakes appear in clusters?
Did they happen during Poor-or-worse days?
Were they inside the train period or the test period?
```

For Asha, the date matters because a large error during an ordinary day is different from a large error before a dangerous pollution day.

---

### 💡 The idea

Take the same persistence error from Chart 3:

```text
persistence error
    = |tomorrow's actual PM2.5 - today's prediction|
    = |target - pm25_until_17|
```

Then place that error on a timeline.

Instead of plotting:

```text
today's PM2.5  →  tomorrow's PM2.5
```

we now plot:

```text
date  →  size of persistence mistake
```

Conceptually:

```text
Persistence error
       │                         ●
       │
       │              ●
       │      ●                 ●
       │  ●
       └────────────────────────────────► Date
          calm days       pollution event
```

The taller the point, the more persistence was wrong on that date.

---

### 🔍 What this chart actually is

This is a **time-series plot of forecast error**.

```text
x-axis = date
y-axis = absolute persistence error in µg/m³
```

Each point means:

```text
On this date, persistence missed tomorrow's PM2.5 by this many µg/m³.
```

For example:

```text
Today's prediction = 60
Tomorrow's actual value = 85

Error = |85 - 60|
Error = 25 µg/m³
```

That day appears at:

```text
x = that date
y = 25
```

The chart is no longer showing the pollution level itself.

It is showing how badly the prediction failed.

---

### How to read the height of the line

#### Low point

```text
Small persistence error
        ↓
Today's value was a reasonable guess for tomorrow.
```

#### High point

```text
Large persistence error
        ↓
Tomorrow changed considerably from today.
```

#### Cluster of high points

```text
Several large errors close together
        ↓
A difficult pollution window or event.
```

#### One isolated high point

```text
One unusual change
        ↓
A possible one-day event or measurement change.
```

The visual meaning is:

```text
short line/point
    → persistence worked reasonably well

tall spike
    → persistence failed badly

several tall spikes together
    → persistence struggled during a period, not just one day
```

---

### What the red `x` markers mean

The red `x` markers identify dates where:

```text
tomorrow's target >= 91 µg/m³
```

That means tomorrow was classified as **Poor-or-worse**.

The chart combines two pieces of information:

```text
Blue line
    → how large the persistence error was

Red x marker
    → tomorrow was a Poor-or-worse day
```

This lets us ask:

```text
Do the largest errors happen when tomorrow's air becomes dangerous?
```

Visually:

```text
Large blue spike + red x
        ↓
Persistence made a large error before a Poor-or-worse day.
```

That is especially important for Asha because an error before a dangerous day may lead to a missed warning.

---

### The visual story

The chart is trying to reveal this pattern:

```text
ordinary days
    → small errors
    → persistence is reliable

start or end of a pollution event
    → large errors
    → today's value is no longer a safe guide to tomorrow

Poor-or-worse days
    → red markers
    → days where the forecast decision matters more
```

A conceptual chart:

```text
Error
  │
  │                         ●
  │                         ×  ← tomorrow was Poor-or-worse
  │              ●
  │       ●      ×
  │  ●    ×
  │────────────────────────────────────
  └────────────────────────────────────► Date
      ordinary days       pollution event
```

The important question is not only:

```text
How large is the error?
```

It is also:

```text
When does the error become large?
```

---

### The error formula

The code calculates:

\[
e_i = |y_i - x_i|
\]

Where:

- \(e_i\) — absolute persistence error for day \(i\), measured in µg/m³
- \(y_i\) — tomorrow's actual PM2.5 value
- \(x_i\) — today's PM2.5 value used as the prediction
- \(i\) — one scored date
- \(|\ |\) — absolute value, so positive and negative mistakes are treated as equally large

Worked example:

```text
Today's prediction       = 70
Tomorrow's actual value  = 120
```

```text
e = |120 - 70|
e = |50|
e = 50 µg/m³
```

This creates a 50 µg/m³ spike on that date.

If pollution instead falls:

```text
Today's prediction       = 120
Tomorrow's actual value  = 70
```

```text
e = |70 - 120|
e = |-50|
e = 50 µg/m³
```

The chart shows the same error size.

It does not show the direction of the mistake. It only shows how far persistence was from reality.

---

### Why use absolute error here?

The purpose of this chart is to find the days when persistence failed most severely.

We care about the size of the miss:

```text
prediction too low by 50
prediction too high by 50
```

Both are serious 50 µg/m³ mistakes.

Using the absolute value makes both appear as:

```text
error = 50 µg/m³
```

This makes the largest failures easy to identify as the tallest spikes.

---

### What the code does visually

The code follows this flow:

```text
daily_scoring
        │
        ▼
target - pm25_until_17
        │
        ▼
Take the absolute value
        │
        ▼
Store it as persistence_error
        │
        ▼
Plot error against date
        │
        ▼
Mark Poor-or-worse target days with red x markers
        │
        ▼
Mark the train/test boundary
```

The important mappings are:

```text
daily_scoring.index
    ──► x-axis: date

daily_scoring["persistence_error"]
    ──► y-axis: error size

poor_days.index
    ──► dates receiving red x markers

cut_date
    ──► vertical train/test boundary
```

The printed values compare three groups:

```text
Mean error on all scoring rows
    → typical error across all usable dates

Mean error on Poor-or-worse days
    → typical error when tomorrow's pollution is severe

Mean error on ordinary days
    → typical error when tomorrow is below the threshold
```

This comparison answers:

```text
Is persistence less reliable before dangerous pollution days?
```

---

### What the vertical train/test line means

The dotted vertical line marks:

```text
2026-03-01
```

It separates:

```text
Before 2026-03-01
    → training history

From 2026-03-01 onward
    → future test period
```

This allows us to check whether large errors are:

```text
only historical,
or also present in the unseen test period.
```

If large spikes appear in the test period, the problem is still present in future-like data.

That matters more than finding a problem only in old history.

---

### 🤔 Before you run this

Predict what the chart will show:

```text
Will persistence errors:

(a) remain almost the same every day?

(b) appear as occasional tall spikes?

(c) be largest only during low-pollution days?

(d) become larger around Poor-or-worse days?
```

Also predict:

```text
Will the mean error on Poor-or-worse days be higher
or lower than the mean error on ordinary days?
```

Run the cell and compare your prediction with the result.

---

### 👀 What to look at here

First, look for the tallest blue spikes. Hover over them to identify their dates and exact error sizes.

Next, check whether the tall spikes also have red `x` markers. A red marker means tomorrow was Poor-or-worse, so a nearby spike may represent a missed dangerous change.

Finally, compare the three printed means. If the Poor-or-worse mean is higher than the ordinary-day mean, persistence is less dependable when the forecast matters most.

---

### What this chart does not prove

The chart can show that large errors occur near certain dates.

It cannot prove the exact cause of those errors.

Possible causes include:

```text
Diwali or other events
crop-burning season
weather changes
wind changes
dust
traffic
industrial activity
sensor behaviour
```

A date is a clue, not proof of causation.

We need an additional investigation to connect a date with a likely real-world cause.

---

### ⚠️ Small documentation correction

The explanation describes the red markers as a “rug” along the top.

However, the current code does not create a separate rug plot.

It places red `x` markers directly on the persistence-error line:

```python
chart4.add_trace(go.Scatter(...))
```

Therefore, the accurate description is:

```text
Red x markers identify dates when tomorrow was Poor-or-worse.
```

The visual meaning is still clear; only the wording should be corrected.

---

### ✅ Checkpoint

1. What does a tall spike in the chart represent?

2. What does a red `x` marker represent?

3. Why is a timeline more useful here than the scatter plot from Chart 3?

---

### Answers

1. It represents a date when persistence made a large prediction error.

2. It represents a date where tomorrow's target was Poor-or-worse, meaning `target >= 91`.

3. Chart 3 shows the size and shape of the relationship but does not show when each error happened. The timeline connects every error to a real calendar date.

---

### Final takeaway

Chart 3 showed:

```text
How far each prediction was from the diagonal.
```

Chart 4 shows:

```text
When those mistakes happened.
```

The visual flow is:

```text
today's PM2.5
        ↓
used as tomorrow's prediction
        ↓
compare with tomorrow's actual value
        ↓
calculate the absolute error
        ↓
place that error on its date
        ↓
look for spikes and clusters
```

The main visual conclusions are:

```text
Small errors
    → persistence worked reasonably well

Tall spikes
    → persistence failed badly

Clusters of spikes
    → difficult pollution window

Red x on a spike
    → large miss before a Poor-or-worse day

Higher Poor-or-worse mean error
    → persistence is weakest when the decision is most important
```

This gives DAF-14 and DAF-18 a practical direction:

> Persistence should not be trusted equally on every day. Event-aware features and safer decision rules are most valuable around the dates where the error spikes.

The next step is:

> **Find the ten worst persistence-error days and investigate a likely cause for each.**

In [5]:
# Step 5: Chart 4 -- persistence error over time
daily_scoring["persistence_error"] = (daily_scoring["target"] - daily_scoring["pm25_until_17"]).abs()

chart4 = go.Figure()

chart4.add_trace(go.Scatter(
    x=daily_scoring.index, y=daily_scoring["persistence_error"],
    mode="lines+markers", name="Persistence error",
    line=dict(color="#4C78A8", width=1),
    marker=dict(size=4),
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Error: %{y:.1f}<extra></extra>",
))

poor_days = daily_scoring[daily_scoring["poor_or_worse"]]
chart4.add_trace(go.Scatter(
    x=poor_days.index, y=poor_days["persistence_error"],
    mode="markers", name="Tomorrow was Poor-or-worse",
    marker=dict(size=8, color="firebrick", symbol="x"),
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Error: %{y:.1f}<extra></extra>",
))

chart4.add_vline(
    x=cut_date, line_dash="dot", line_color="gray",
    annotation_text="train | test cut", annotation_position="top right",
)

chart4.update_layout(
    title="On which days does persistence fail badly?",
    xaxis_title="Date", yaxis_title="Persistence error (µg/m³)",
    height=450,
)
chart4.show()

print("Mean persistence error (all scoring rows):", round(daily_scoring["persistence_error"].mean(), 2))
print("Mean persistence error on Poor-or-worse days:", round(poor_days["persistence_error"].mean(), 2))
print("Mean persistence error on ordinary days:", round(daily_scoring.loc[~daily_scoring['poor_or_worse'], 'persistence_error'].mean(), 2))


Mean persistence error (all scoring rows): 27.31
Mean persistence error on Poor-or-worse days: 54.17
Mean persistence error on ordinary days: 15.06


### Takeaway

The biggest misses are not evenly spread — they cluster in short bursts
around known high-pollution windows (the October Diwali/crop-burning stretch
stands out sharply), and the average error on Poor-or-worse days is noticeably
higher than on ordinary days. So persistence doesn't fail randomly: it fails
in a *predictable kind of moment* — right as a spike starts or ends, when
"tomorrow = today" is the worst possible assumption. **This is exactly the
evidence DAF-14 needs to justify event-aware features (e.g. days-since-Diwali,
a crop-burning-season flag), and DAF-18 needs it to decide that persistence
alone is not safe to act on during those windows.**

**Next: Step 6 — find the ten worst persistence-error days and look up a
likely cause for each.**


## Step 6: the ten worst persistence-error days, and why

### 🎬 How we got here

Chart 4 showed persistence errors clustering around known bad windows rather
than being random. Now we pin down the actual worst ten days by number, and
look up what really happened outside the dataset — the sensor only tells us
*that* PM2.5 swung hard, not *why*.

### 💡 The idea

Sort `daily_scoring` by `persistence_error` and take the top ten rows. Each
row's index is "today" (the day persistence looked at); the miss is about
**tomorrow**, so the day that actually matters for the cause is `today + 1`:

```text
row index (today)   pm25_until_17   target (tomorrow)   error
   2025-10-20            230              657             428   ← tomorrow = Oct 21
                                                                     (Diwali night)
```

We then check each cause against real, verifiable events — Diwali/fireworks
dates, IMD dust-storm reports, and the CSE/CPCB winter-smog reporting for
Delhi — rather than guessing from the number alone.


# 🟨 Explanation_2 — Step 6: Find the ten worst persistence-error days

<!-- filepath: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/notebooks/11_eda.ipynb -->

### 🎬 How we got here

Chart 4 showed the persistence error as a timeline.

We saw that most days had small errors, but some dates produced very tall spikes.

Chart 4 answered:

```text
When did persistence fail badly?
```

Now we want to identify the exact dates behind those spikes.

---

### 😖 The problem

A visual spike tells us that something went wrong:

```text
This date had a very large prediction error.
```

But the chart alone does not immediately tell us:

```text
What was today's value?
What actually happened tomorrow?
How large was the error?
Did several bad dates belong to the same event?
What might have caused the sudden change?
```

We need to convert the tallest chart spikes into a small, readable table.

---

### 💡 The idea

Sort all scoring rows from the largest persistence error to the smallest.

Then keep only the first ten rows.

Conceptually:

```text
All scoring days
        │
        ▼
Sort by persistence_error, largest first
        │
        ▼
Keep the top 10 rows
        │
        ▼
Inspect the dates and values
        │
        ▼
Look for a likely external explanation
```

This is similar to checking the top ten slowest API requests after a performance incident.

The analogy breaks down because these are pollution events, not software failures. A large error is a clue that needs investigation, not automatically a confirmed root cause.

---

### 🔍 What this actually is

This is a **top-N error investigation**.

```text
N = 10
```

The code selects the ten rows with the largest value of:

```text
persistence_error
```

Each selected row represents:

```text
row date          = today
pm25_until_17    = today's prediction
target           = tomorrow's actual PM2.5
persistence_error = size of the mistake
```

The table is read like this:

```text
Today ───────────────► prediction is made
Tomorrow ────────────► actual result becomes known
Difference ──────────► persistence error
```

For example:

```text
Today:                  2025-10-20
Today's prediction:     230 µg/m³
Tomorrow's actual:      657 µg/m³
Error:                  427 µg/m³
```

The important date for the pollution event is:

```text
2025-10-21
```

That is why the code creates:

```python
tomorrow_date = today + 1 day
```

---

### Why is the row date called “today”?

The row index is the day on which persistence made its prediction.

```text
row index = today
target    = tomorrow's actual value
```

Visual timeline:

```text
2025-10-20                         2025-10-21
today's information                tomorrow's result
       │                                   │
       ├── predict 230 ───────────────────►│
       │                                   ├── actual = 657
       │                                   │
       └──────────── error = 427 ──────────┘
```

So:

```text
The row date identifies the prediction day.

The tomorrow date identifies the day whose pollution outcome we investigate.
```

This distinction prevents us from blaming the wrong calendar date.

---

### What the output table means visually

The output table has four important columns:

```text
today
    → date when persistence made the prediction

tomorrow_actual
    → actual PM2.5 value on the next day

persistence_error
    → absolute size of the mistake

tomorrow_date
    → date to investigate for a possible cause
```

A conceptual output looks like this:

```text
┌────────────┬────────────────┬───────────────────┬───────────────┐
│ today      │ tomorrow_actual│ persistence_error │ tomorrow_date │
├────────────┼────────────────┼───────────────────┼───────────────┤
│ Oct 20     │ 657            │ 428               │ Oct 21        │
│ Oct 21     │ 235            │ 538               │ Oct 22        │
│ Oct 19     │ 230            │ 190               │ Oct 20        │
└────────────┴────────────────┴───────────────────┴───────────────┘
```

The largest error appears at the top.

The table is therefore a ranked list of the dates where:

```text
tomorrow changed most dramatically compared with today.
```

---

### The error calculation

Persistence predicts:

\[
\hat{y}_i = x_i
\]

Where:

- \(\hat{y}_i\) — persistence prediction for day \(i\), measured in µg/m³
- \(x_i\) — today's `pm25_until_17` value, measured in µg/m³
- \(i\) — one scored date

The actual result is:

\[
y_i = \text{target}_i
\]

The absolute error is:

\[
e_i = |y_i - \hat{y}_i|
\]

Where:

- \(e_i\) — size of the prediction mistake, measured in µg/m³
- \(y_i\) — tomorrow's actual PM2.5 value
- \(\hat{y}_i\) — persistence prediction
- \(|\ |\) — absolute value, which removes the direction

Worked example:

```text
Today's prediction       = 230
Tomorrow's actual value  = 657
```

```text
e = |657 - 230|
e = |427|
e = 427 µg/m³
```

The row is ranked according to this `427` value.

If pollution falls instead:

```text
Today's prediction       = 773
Tomorrow's actual value  = 235
```

```text
e = |235 - 773|
e = |-538|
e = 538 µg/m³
```

Both a sudden increase and a sudden decrease can appear in the top ten.

---

### How the sorting works visually

Before sorting:

```text
Date A → error 12
Date B → error 538
Date C → error 45
Date D → error 204
```

After sorting from largest to smallest:

```text
1. Date B → error 538
2. Date D → error 204
3. Date C → error 45
4. Date A → error 12
```

Then `.head(10)` keeps only:

```text
the ten tallest error spikes
```

The code does not recalculate the error.

It uses the `persistence_error` column already created by Chart 4.

---

### What the code does

The code follows this sequence:

```text
daily_scoring
        │
        ▼
Sort rows by persistence_error
        │
        ▼
Place the largest errors first
        │
        ▼
Keep the first 10 rows
        │
        ▼
Copy those rows into worst10
        │
        ▼
Add one day to each row index
        │
        ▼
Create tomorrow_date
        │
        ▼
Print the investigation table
```

The important code mappings are:

```text
sort_values("persistence_error", ascending=False)
    ──► largest mistakes first

.head(10)
    ──► keep the ten worst mistakes

worst10.index + pd.Timedelta(days=1)
    ──► calculate the date of tomorrow's actual outcome

.rename(...)
    ──► make the printed table easier to read
```

The variable:

```python
worst10
```

is a smaller DataFrame containing only the ten most serious persistence failures.

---

### Why investigate the top ten?

The top ten are not necessarily a representative sample of every day.

They are deliberately selected because they are the most extreme failures.

This makes them useful for:

```text
finding unusual events
finding repeated event windows
finding missing features
finding possible sensor problems
deciding what future data to collect
```

The purpose is not to claim:

```text
These ten days describe the whole dataset.
```

The purpose is to ask:

```text
What kind of situation creates the largest failures?
```

---

### How to read the dates as groups

Do not inspect only one row at a time.

Look for dates that sit close together:

```text
Oct 19
Oct 20
Oct 21
Oct 22
```

This pattern suggests one continuous event window rather than four unrelated failures.

Visual grouping:

```text
Error
  │          ●
  │       ●  ●
  │    ●
  │
  └────┬────┬────┬────────────────────► Date
      Oct19 Oct20 Oct21 Oct22
       one connected event window
```

Other possible patterns:

```text
Several October dates
    → possible festival or seasonal window

Several May dates
    → possible weather or dust event

Several December dates
    → possible stagnant winter-air period

One isolated date
    → possible one-day event, data issue, or sudden change
```

These are investigation hypotheses, not automatic conclusions.

---

### From error spike to likely cause

The table gives us an observation:

```text
PM2.5 changed sharply between two consecutive days.
```

External evidence is then used to investigate the reason.

The reasoning flow is:

```text
Large persistence error
        ↓
Identify tomorrow_date
        ↓
Group nearby worst dates
        ↓
Check known events and reports
        ↓
Compare the event date with the pollution spike
        ↓
Record a likely cause with supporting evidence
```

Possible explanations include:

```text
festival fireworks
dust storm
stagnant winter air
crop-burning period
rain or strong wind
traffic or industrial activity
sensor behaviour
```

A cause should be called **likely** only when the date agrees with an external source or a clearly documented event.

---

### Important difference: observation versus explanation

The sensor data can directly show:

```text
Today's PM2.5 was different from tomorrow's PM2.5.
```

The sensor data alone cannot prove:

```text
Diwali caused the change.
A dust storm caused the change.
Stagnant air caused the change.
```

Visual evidence:

```text
What the table proves
    ├── the date
    ├── today's value
    ├── tomorrow's value
    └── the error size

What external investigation may explain
    └── why the value changed
```

This is the difference between:

```text
observation = what happened
cause       = why it happened
```

Do not turn a date match into proof of causation without supporting evidence.

---

### What this step does not do

This step does not:

```text
fit a machine-learning model
improve the forecast
prove causation
prove the sensor is correct
prove that every future event will repeat
```

It creates a focused investigation list.

The next analysis must validate the likely causes using reliable external information.

---

### 🤔 Before you run this

Predict what the output will show:

```text
After sorting by persistence_error:

(a) the smallest errors will appear first

(b) the largest errors will appear first

(c) the rows will remain in calendar order

(d) all rows will have the same error
```

Also predict:

```text
If several top-ten dates are consecutive,
does that suggest one event window or many unrelated events?
```

Run the cell and compare your prediction with the table.

---

### 👀 What to look at here

First, look at the `persistence_error` column. The first row should contain the largest miss in the scoring data.

Next, compare `today` with `tomorrow_date`. The external event investigation should focus on `tomorrow_date`, because that is when the actual pollution outcome occurred.

Finally, look for nearby dates in the top ten. Consecutive dates suggest a connected pollution window, while isolated dates need separate investigation.

---

### ✅ Checkpoint

1. Why is `tomorrow_date` one day after the row index?

2. What does a large `persistence_error` tell us?

3. Does a date match prove that an external event caused the pollution spike?

---

### Answers

1. The row index is the day when persistence made the prediction. The target belongs to the following day, so the event date is `today + 1 day`.

2. Tomorrow's actual PM2.5 was very different from today's value used as the prediction.

3. No. It is evidence worth investigating, but the cause needs supporting external evidence.

---

### Final takeaway

Step 6 converts the tallest visual spikes from Chart 4 into a ranked investigation table:

```text
Chart 4
    → shows when persistence failed

Step 6
    → identifies the ten largest failures

tomorrow_date
    → tells us which day to investigate

external evidence
    → helps explain what may have caused the change
```

The key visual idea is:

```text
one large error
    → one clue

several nearby large errors
    → one possible event window

repeated event windows
    → possible missing feature or missing data source
```

This tells DAF-14 and DAF-18 where persistence needs help:

```text
festival-aware features
weather or dust signals
wind and stagnation information
safer decisions around known event windows
```

The most important lesson is:

> The sensor tells us **what changed**. The ranked top-ten table tells us **when to investigate**. External evidence is needed to understand **why it changed**.

In [6]:
# Step 6: the ten worst persistence-error days
worst10 = daily_scoring.sort_values("persistence_error", ascending=False).head(10).copy()
worst10["tomorrow_date"] = worst10.index + pd.Timedelta(days=1)

print(worst10[["pm25_until_17", "target", "persistence_error", "tomorrow_date"]]
      .rename(columns={"pm25_until_17": "today", "target": "tomorrow_actual"})
      .to_string())


                                today  tomorrow_actual  persistence_error             tomorrow_date
2025-10-21 00:00:00+05:30  772.777778       235.083333         537.694444 2025-10-22 00:00:00+05:30
2025-10-20 00:00:00+05:30  229.666667       657.291667         427.625000 2025-10-21 00:00:00+05:30
2025-05-15 00:00:00+05:30  307.666667        91.208333         216.458333 2025-05-16 00:00:00+05:30
2025-05-14 00:00:00+05:30   46.333333       250.750000         204.416667 2025-05-15 00:00:00+05:30
2025-12-12 00:00:00+05:30  199.333333       397.375000         198.041667 2025-12-13 00:00:00+05:30
2025-10-19 00:00:00+05:30  145.222222       335.238095         190.015873 2025-10-20 00:00:00+05:30
2025-12-17 00:00:00+05:30  170.611111       345.562500         174.951389 2025-12-18 00:00:00+05:30
2025-12-23 00:00:00+05:30  300.611111       126.875000         173.736111 2025-12-24 00:00:00+05:30
2025-10-31 00:00:00+05:30  106.222222       279.833333         173.611111 2025-11-01 00:00:00+05:30


### What happened on each of these days

Looked up against news and CPCB/IMD reporting for R K Puram / Delhi NCR:

| Today (row) | Tomorrow (the miss) | Error | Likely cause |
|---|---|---:|---|
| 2025-10-20 | 2025-10-21 | 428 | **Diwali** — main celebrations fell on Oct 20, with fireworks continuing past permitted hours; PM2.5 in Delhi hit its worst Diwali level in 5 years (~488 µg/m³ average), overnight spike |
| 2025-10-21 | 2025-10-22 | 538 | Diwali comedown — the extreme fireworks pollution from the night of the 20th/21st clears unevenly the next day (Govardhan Puja, Oct 22) |
| 2025-10-19 | 2025-10-20 | 190 | Choti Diwali (Oct 19) fireworks build-up ahead of main Diwali |
| 2025-10-31 | 2025-11-01 | 174 | Post-Diwali stubble-burning season ramping up into November, reported "Poor" AQI category around this date |
| 2025-05-14 | 2025-05-15 | 204 | **Dust storm** — IMD-reported dust advected from north Pakistan via Punjab/Haryana, visibility fell from 4,500 m to 1,200 m overnight; AQI jumped from ~135 (moderate) to ~289–310 (poor) the next day, RK Puram specifically at 310 |
| 2025-05-15 | 2025-05-16 | 216 | Same dust-storm event clearing out as forecast thunderstorms/rain arrived, air improving sharply the day after |
| 2025-12-01 | 2025-12-02 | 155 | Early-December stagnant-air winter smog window (CSE: December 2025 had more intense smog than the Oct/Nov stubble period) |
| 2025-12-12 | 2025-12-13 | 198 | Same winter stagnant-air pattern — wind calm, local + regional emissions trapped |
| 2025-12-17 | 2025-12-18 | 175 | Same winter stagnant-air pattern, days ahead of the season's worst reading on Dec 14 easing and swinging again |
| 2025-12-23 | 2025-12-24 | 174 | Same winter stagnant-air pattern, late-December smog swings reported alongside Ghaziabad/Greater Noida also above 300 µg/m³ |

Three real causes explain all ten days: **the Diwali fireworks window
(Oct 19–22)**, **one dust-storm event (May 14–16)**, and **December's
stagnant-air winter smog**, which the CSE analysis found was *more* intense
than the stubble-burning period itself, not caused by it.

### Takeaway

Every one of the ten worst misses has a real, nameable external cause — none
of them are noise or bad sensor data. That is useful and a little
uncomfortable: **DAF-14 could add a Diwali/festival flag and a rough
dust-storm signal (e.g. a sudden-wind-speed proxy), which would catch the
May and October misses. But the December cluster is driven by day-to-day
wind stagnation that isn't visible in this table at all — no wind or weather
column exists in `daily_17.parquet` yet.** That's a real gap for DAF-14 and
DAF-18 to weigh: either accept persistence will keep missing stagnant-smog
swings, or a future ticket needs to bring in an external weather feed.

**This closes the DAF-11 chart work. Next: fill in the ticket's *My notes*
and *Explain back* answers.**


## DAF-11 at a glance

<div style="background:#F6F3EC; color:#211E19; font-family:'IBM Plex Sans', -apple-system, sans-serif; border:1.5px solid #CFC7B2; border-radius:16px; padding:20px 20px 24px; max-width:940px; margin:0 auto;">
<div style="display:flex; justify-content:space-between; align-items:center; gap:12px; border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:12px 16px; margin-bottom:16px; flex-wrap:wrap;">
  <span style="font-family:'IBM Plex Mono',monospace; font-weight:600; font-size:12px; color:#5C574C; letter-spacing:.04em;">DAF-11</span>
  <span style="font-weight:700; font-size:17px; text-align:center;">EDA on the Daily Table &mdash; the story in four charts</span>
  <span style="font-family:'IBM Plex Mono',monospace; font-size:11px; font-weight:600; color:#3E6E52; background:rgba(62,110,82,.12); border:1px solid rgba(62,110,82,.4); border-radius:999px; padding:4px 10px;">&#10003; DONE</span>
</div>
<div style="display:flex; gap:14px; flex-wrap:wrap; margin-bottom:18px;">
  <div style="flex:1 1 260px; border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:14px 16px;">
    <div style="font-family:'IBM Plex Mono',monospace; font-size:11px; font-weight:700; letter-spacing:.08em; color:#C77E1F; text-transform:uppercase; margin-bottom:10px;">&#127919; Goal</div>
    <svg viewBox="0 0 260 90" width="100%" height="90" role="img" aria-label="Four charts, each an arrow into a ticket, no model fitted">
      <g font-family="IBM Plex Mono, monospace" font-size="9" fill="#211E19">
        <rect x="4" y="6" width="60" height="20" rx="4" fill="none" stroke="#211E19" stroke-width="1.2"/>
        <text x="34" y="19" text-anchor="middle">4 charts</text>
        <line x1="64" y1="16" x2="90" y2="16" stroke="#211E19" stroke-width="1.2" marker-end="url(#a1)"/>
        <rect x="92" y="6" width="60" height="20" rx="4" fill="none" stroke="#B5402C" stroke-width="1.2"/>
        <text x="122" y="19" text-anchor="middle" fill="#B5402C">1 ticket each</text>
        <rect x="4" y="34" width="80" height="20" rx="4" fill="none" stroke="#211E19" stroke-width="1.2"/>
        <text x="44" y="47" text-anchor="middle">10 worst days</text>
        <line x1="84" y1="44" x2="110" y2="44" stroke="#211E19" stroke-width="1.2" marker-end="url(#a1)"/>
        <rect x="112" y="34" width="70" height="20" rx="4" fill="none" stroke="#B5402C" stroke-width="1.2"/>
        <text x="147" y="47" text-anchor="middle" fill="#B5402C">real cause</text>
        <rect x="4" y="62" width="120" height="20" rx="4" fill="none" stroke="#5C574C" stroke-width="1.2" stroke-dasharray="3 2"/>
        <text x="64" y="75" text-anchor="middle" fill="#5C574C">train + test, no fit()</text>
      </g>
      <defs><marker id="a1" viewBox="0 0 8 8" refX="6" refY="4" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#211E19"/></marker></defs>
    </svg>
  </div>
  <div style="flex:1 1 260px; border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:14px 16px;">
    <div style="font-family:'IBM Plex Mono',monospace; font-size:11px; font-weight:700; letter-spacing:.08em; color:#3E6E52; text-transform:uppercase; margin-bottom:10px;">&#10003; Result</div>
    <svg viewBox="0 0 260 90" width="100%" height="90" role="img" aria-label="Four of four charts done, three tickets fed, ten of ten days explained, one gap found">
      <g font-family="IBM Plex Mono, monospace" font-size="11" font-weight="700">
        <text x="10" y="20" fill="#3E6E52">4/4</text>
        <text x="45" y="20" fill="#211E19" font-weight="400" font-size="9.5">charts shipped</text>
        <text x="150" y="20" fill="#3E6E52">3</text>
        <text x="163" y="20" fill="#211E19" font-weight="400" font-size="9.5">tickets fed</text>
        <text x="10" y="45" fill="#3E6E52">10/10</text>
        <text x="55" y="45" fill="#211E19" font-weight="400" font-size="9.5">days explained</text>
        <text x="150" y="45" fill="#B5402C">1</text>
        <text x="163" y="45" fill="#211E19" font-weight="400" font-size="9.5">gap found</text>
      </g>
      <rect x="8" y="58" width="244" height="14" rx="7" fill="none" stroke="#CFC7B2" stroke-width="1.2"/>
      <rect x="8" y="58" width="244" height="14" rx="7" fill="#3E6E52"/>
      <text x="130" y="68" text-anchor="middle" font-size="8.5" fill="#fff" font-family="IBM Plex Mono, monospace">DAF-11 complete</text>
    </svg>
  </div>
</div>
<table style="width:100%; border-collapse:collapse;">
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #211E19; background:#FFFFFF; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">1</div>
  </td>
  <td style="padding:0 0 14px 10px;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">Rebuild the DAF-06 split</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#5C574C;">setup</span>
      </div>
      <svg viewBox="0 0 400 70" width="100%" height="70" role="img" aria-label="571 days split at 2026-03-01 into 319 train days and 168 test days">
        <rect x="10" y="20" width="380" height="26" fill="none" stroke="#CFC7B2" stroke-width="1"/>
        <rect x="10" y="20" width="262" height="26" fill="#EFE9D9"/>
        <rect x="272" y="20" width="118" height="26" fill="rgba(181,64,44,.15)"/>
        <line x1="272" y1="10" x2="272" y2="56" stroke="#211E19" stroke-width="1.4" stroke-dasharray="3 2"/>
        <text x="141" y="37" text-anchor="middle" font-size="11" font-family="IBM Plex Mono, monospace" font-weight="600">train &middot; 319 days</text>
        <text x="331" y="37" text-anchor="middle" font-size="11" font-family="IBM Plex Mono, monospace" font-weight="600" fill="#B5402C">test &middot; 168 days</text>
        <text x="272" y="8" text-anchor="middle" font-size="9.5" font-family="IBM Plex Mono, monospace" fill="#5C574C">cut: 2026-03-01</text>
        <text x="10" y="64" font-size="9" fill="#5C574C">2025-02-19</text>
        <text x="390" y="64" text-anchor="end" font-size="9" fill="#5C574C">2026-09-12</text>
      </svg>
    </div>
  </td>
</tr>
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #211E19; background:#FFFFFF; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">2</div>
  </td>
  <td style="padding:0 0 14px 10px;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">Chart 1 &mdash; when is the decision actually hard?</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#B5402C; background:rgba(181,64,44,.12); border:1px solid rgba(181,64,44,.35); border-radius:999px; padding:2px 8px;">&#8594; DAF-16</span>
      </div>
      <svg viewBox="0 0 400 110" width="100%" height="110" role="img" aria-label="Daily PM2.5 line chart with a Poor threshold line at 91; points above it clustered in winter are marked red, 30.8 percent of all days">
        <line x1="30" y1="18" x2="380" y2="18" stroke="#B5402C" stroke-width="1.2" stroke-dasharray="4 3"/>
        <text x="378" y="32" text-anchor="end" font-size="9" font-family="IBM Plex Mono, monospace" fill="#B5402C">Poor line: 91</text>
        <polyline points="30,88 55,82 80,85 105,40 120,22 135,38 160,80 185,86 210,60 225,26 240,52 265,84 290,88 315,60 330,30 345,46 370,82"
                  fill="none" stroke="#211E19" stroke-width="1.6"/>
        <circle cx="120" cy="22" r="3.2" fill="#B5402C"/>
        <circle cx="225" cy="26" r="3.2" fill="#B5402C"/>
        <circle cx="330" cy="30" r="3.2" fill="#B5402C"/>
        <line x1="30" y1="98" x2="380" y2="98" stroke="#CFC7B2" stroke-width="1"/>
        <text x="30" y="108" font-size="9" fill="#5C574C">2025-02</text>
        <text x="370" y="108" text-anchor="end" font-size="9" fill="#5C574C">2026-09</text>
        <text x="200" y="10" text-anchor="middle" font-size="12" font-family="IBM Plex Mono, monospace" font-weight="700" fill="#B5402C">30.8% of days above the Poor line</text>
      </svg>
    </div>
  </td>
</tr>
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #211E19; background:#FFFFFF; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">3</div>
  </td>
  <td style="padding:0 0 14px 10px;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">Chart 2 &mdash; how strong is the season, month by month?</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#B5402C; background:rgba(181,64,44,.12); border:1px solid rgba(181,64,44,.35); border-radius:999px; padding:2px 8px;">&#8594; DAF-14 calendar</span>
      </div>
      <svg viewBox="0 0 400 110" width="100%" height="110" role="img" aria-label="Box plots by month, tall boxes in Oct through Feb, short boxes in Jun through Sep">
        <line x1="20" y1="90" x2="390" y2="90" stroke="#CFC7B2" stroke-width="1"/>
        <line x1="20" y1="90" x2="20" y2="20" stroke="#CFC7B2" stroke-width="1"/>
        <g stroke="#211E19" stroke-width="1.3" fill="none">
          <rect x="30" y="66" width="18" height="16"/><line x1="39" y1="58" x2="39" y2="66"/><line x1="39" y1="82" x2="39" y2="88"/>
          <rect x="58" y="70" width="18" height="12"/><line x1="67" y1="64" x2="67" y2="70"/><line x1="67" y1="82" x2="67" y2="86"/>
          <rect x="86" y="72" width="18" height="10"/><line x1="95" y1="67" x2="95" y2="72"/><line x1="95" y1="82" x2="95" y2="85"/>
          <rect x="114" y="76" width="18" height="8"/><line x1="123" y1="72" x2="123" y2="76"/><line x1="123" y1="84" x2="123" y2="87"/>
          <rect x="142" y="78" width="18" height="7"/><line x1="151" y1="74" x2="151" y2="78"/><line x1="151" y1="85" x2="151" y2="88"/>
          <rect x="170" y="80" width="18" height="6" stroke="#3E6E52"/><line x1="179" y1="77" x2="179" y2="80" stroke="#3E6E52"/><line x1="179" y1="86" x2="179" y2="88" stroke="#3E6E52"/>
          <rect x="198" y="30" width="18" height="34" stroke="#B5402C"/><line x1="207" y1="22" x2="207" y2="30" stroke="#B5402C"/><line x1="207" y1="64" x2="207" y2="74" stroke="#B5402C"/>
          <rect x="226" y="24" width="18" height="36" stroke="#B5402C"/><line x1="235" y1="16" x2="235" y2="24" stroke="#B5402C"/><line x1="235" y1="60" x2="235" y2="70" stroke="#B5402C"/>
          <rect x="254" y="56" width="18" height="22"/><line x1="263" y1="46" x2="263" y2="56"/><line x1="263" y1="78" x2="263" y2="84"/>
          <rect x="282" y="68" width="18" height="14"/><line x1="291" y1="60" x2="291" y2="68"/><line x1="291" y1="82" x2="291" y2="86"/>
          <rect x="310" y="72" width="18" height="10"/><line x1="319" y1="66" x2="319" y2="72"/><line x1="319" y1="82" x2="319" y2="85"/>
          <rect x="338" y="74" width="18" height="9"/><line x1="347" y1="69" x2="347" y2="74"/><line x1="347" y1="83" x2="347" y2="86"/>
        </g>
        <text x="207" y="12" text-anchor="middle" font-size="10" font-family="IBM Plex Mono, monospace" font-weight="700" fill="#B5402C">Dec IQR 95.4</text>
        <text x="179" y="100" text-anchor="middle" font-size="10" font-family="IBM Plex Mono, monospace" font-weight="700" fill="#3E6E52">Jul IQR 10.2</text>
        <text x="20" y="102" font-size="8.5" fill="#5C574C">Feb</text>
        <text x="380" y="102" text-anchor="end" font-size="8.5" fill="#5C574C">Sep</text>
      </svg>
    </div>
  </td>
</tr>
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #211E19; background:#FFFFFF; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">4</div>
  </td>
  <td style="padding:0 0 14px 10px;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">Chart 3 &mdash; why is persistence such a strong baseline?</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#B5402C; background:rgba(181,64,44,.12); border:1px solid rgba(181,64,44,.35); border-radius:999px; padding:2px 8px;">&#8594; DAF-14 lags</span>
      </div>
      <svg viewBox="0 0 400 130" width="100%" height="130" role="img" aria-label="Scatter of today versus tomorrow PM2.5 hugging a y equals x line, widening into red points at the high end, correlation 0.775">
        <line x1="30" y1="110" x2="370" y2="15" stroke="#5C574C" stroke-width="1" stroke-dasharray="3 3"/>
        <text x="372" y="12" font-size="9" font-family="IBM Plex Mono, monospace" fill="#5C574C">y = x</text>
        <g fill="#211E19">
          <circle cx="55" cy="98" r="2.6"/><circle cx="75" cy="88" r="2.6"/><circle cx="95" cy="78" r="2.6"/>
          <circle cx="115" cy="70" r="2.6"/><circle cx="135" cy="60" r="2.6"/><circle cx="155" cy="52" r="2.6"/>
          <circle cx="175" cy="44" r="2.6"/><circle cx="100" cy="60" r="2.6"/><circle cx="140" cy="76" r="2.6"/>
          <circle cx="200" cy="40" r="2.6"/><circle cx="220" cy="60" r="2.6"/>
        </g>
        <circle cx="290" cy="24" r="4" fill="#B5402C"/>
        <circle cx="260" cy="70" r="4" fill="#B5402C"/>
        <circle cx="320" cy="50" r="4" fill="#B5402C"/>
        <text x="18" y="20" font-size="9" fill="#5C574C">tomorrow</text>
        <text x="330" y="122" font-size="9" fill="#5C574C">today</text>
        <line x1="30" y1="112" x2="370" y2="112" stroke="#CFC7B2" stroke-width="1"/>
        <line x1="30" y1="112" x2="30" y2="15" stroke="#CFC7B2" stroke-width="1"/>
        <text x="200" y="128" text-anchor="middle" font-size="12" font-family="IBM Plex Mono, monospace" font-weight="700" fill="#3E6E52">correlation 0.775</text>
      </svg>
    </div>
  </td>
</tr>
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #211E19; background:#FFFFFF; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">5</div>
  </td>
  <td style="padding:0 0 14px 10px;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">Chart 4 &mdash; on which days does persistence fail badly?</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#B5402C; background:rgba(181,64,44,.12); border:1px solid rgba(181,64,44,.35); border-radius:999px; padding:2px 8px;">&#8594; DAF-14 / DAF-18</span>
      </div>
      <svg viewBox="0 0 400 110" width="100%" height="110" role="img" aria-label="Persistence error over time with sharp spikes on Poor-or-worse days, average error 54.2 versus 15.1 on ordinary days">
        <polyline points="30,88 55,84 80,86 105,78 130,20 145,32 160,80 185,42 210,26 235,86 260,88 285,84 310,30 335,46 360,82"
                  fill="none" stroke="#211E19" stroke-width="1.6"/>
        <g stroke="#B5402C" stroke-width="1.8">
          <line x1="126" y1="15" x2="134" y2="23"/><line x1="134" y1="15" x2="126" y2="23"/>
          <line x1="206" y1="21" x2="214" y2="29"/><line x1="214" y1="21" x2="206" y2="29"/>
          <line x1="306" y1="25" x2="314" y2="33"/><line x1="314" y1="25" x2="306" y2="33"/>
        </g>
        <line x1="30" y1="98" x2="380" y2="98" stroke="#CFC7B2" stroke-width="1"/>
        <text x="30" y="108" font-size="9" fill="#5C574C">2025-02</text>
        <text x="370" y="108" text-anchor="end" font-size="9" fill="#5C574C">2026-09</text>
        <text x="200" y="12" text-anchor="middle" font-size="12" font-family="IBM Plex Mono, monospace" font-weight="700"><tspan fill="#B5402C">54.2</tspan><tspan fill="#211E19"> on Poor days vs </tspan><tspan fill="#3E6E52">15.1</tspan><tspan fill="#211E19"> ordinary</tspan></text>
      </svg>
    </div>
  </td>
</tr>
<tr>
  <td style="width:40px; vertical-align:top; text-align:center; padding-top:2px;">
    <div style="width:32px; height:32px; border-radius:50%; border:2px solid #B5402C; background:#B5402C; color:#fff; display:inline-flex; align-items:center; justify-content:center; font-family:'IBM Plex Mono',monospace; font-weight:700; font-size:13px;">6</div>
  </td>
  <td style="padding:0; vertical-align:top;">
    <div style="border:1.5px solid #CFC7B2; border-radius:12px; background:#FFFFFF; padding:10px 14px;">
      <div style="display:flex; justify-content:space-between; gap:8px; flex-wrap:wrap; margin-bottom:8px;">
        <b style="font-size:14px;">The ten worst misses, named</b>
        <span style="font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:600; color:#5C574C;">closes DAF-11</span>
      </div>
      <svg viewBox="0 0 400 120" width="100%" height="120" role="img" aria-label="Ten worst days split into three causes: six Diwali fireworks days, two dust storm days, four winter smog days">
        <g font-family="IBM Plex Mono, monospace">
          <rect x="20" y="20" width="360" height="26" rx="4" fill="none" stroke="#CFC7B2" stroke-width="1"/>
          <rect x="20" y="20" width="216" height="26" rx="4" fill="rgba(181,64,44,.28)"/>
          <rect x="236" y="20" width="72" height="26" fill="rgba(199,126,31,.28)"/>
          <rect x="308" y="20" width="72" height="26" rx="4" fill="rgba(92,87,76,.22)"/>
          <text x="128" y="37" text-anchor="middle" font-size="11" font-weight="700" fill="#B5402C">6 &middot; Diwali</text>
          <text x="272" y="37" text-anchor="middle" font-size="10" font-weight="700" fill="#C77E1F">2 &middot; dust</text>
          <text x="344" y="37" text-anchor="middle" font-size="10" font-weight="700" fill="#5C574C">4 &middot; smog</text>
          <text x="20" y="60" font-size="9" fill="#5C574C">Oct 19&#8211;22</text>
          <text x="236" y="60" font-size="9" fill="#5C574C">May 14&#8211;16</text>
          <text x="308" y="60" font-size="9" fill="#5C574C">Dec 2025</text>
        </g>
        <g stroke-width="1.6" fill="none">
          <path d="M40 90 l0 -14 M34 82 l12 0 M35 76 l10 8 M45 76 l-10 8" stroke="#B5402C"/>
          <circle cx="272" cy="83" r="9" stroke="#C77E1F"/><path d="M266 83 q6 -6 12 0 M266 83 q6 6 12 0" stroke="#C77E1F"/>
          <path d="M328 90 q8 -14 16 0 q8 -14 16 0" stroke="#5C574C"/>
        </g>
        <text x="200" y="112" text-anchor="middle" font-size="12" font-weight="700" fill="#211E19" font-family="IBM Plex Mono, monospace">10 / 10 misses &mdash; zero unexplained</text>
      </svg>
    </div>
  </td>
</tr>
</table>
<div style="margin-top:16px; border:1.5px solid rgba(181,64,44,.4); border-radius:14px; background:#FFFFFF; padding:14px 16px 16px;">
  <div style="font-family:'IBM Plex Mono',monospace; font-size:11px; font-weight:700; letter-spacing:.08em; color:#B5402C; text-transform:uppercase; margin-bottom:10px;">&#127937; Conclusion</div>
  <svg viewBox="0 0 700 190" width="100%" height="190" role="img" aria-label="Three causes explain all ten misses; Diwali and dust storm are fixable with new features, winter smog needs a missing weather column">
    <g font-family="IBM Plex Mono, monospace">
      <rect x="10" y="15" width="200" height="90" rx="10" fill="#FBF9F4" stroke="#CFC7B2" stroke-width="1.2"/>
      <text x="110" y="40" text-anchor="middle" font-size="20" font-weight="700" fill="#B5402C">6</text>
      <text x="110" y="58" text-anchor="middle" font-size="11" font-weight="600" fill="#211E19">Diwali fireworks</text>
      <text x="110" y="72" text-anchor="middle" font-size="9" fill="#5C574C">Oct 19&#8211;22, 2025</text>
      <text x="110" y="94" text-anchor="middle" font-size="9" fill="#3E6E52" font-weight="700">fixable &#8594; festival flag</text>
      <rect x="250" y="15" width="200" height="90" rx="10" fill="#FBF9F4" stroke="#CFC7B2" stroke-width="1.2"/>
      <text x="350" y="40" text-anchor="middle" font-size="20" font-weight="700" fill="#B5402C">2</text>
      <text x="350" y="58" text-anchor="middle" font-size="11" font-weight="600" fill="#211E19">Dust storm</text>
      <text x="350" y="72" text-anchor="middle" font-size="9" fill="#5C574C">May 14&#8211;16, 2025</text>
      <text x="350" y="94" text-anchor="middle" font-size="9" fill="#3E6E52" font-weight="700">fixable &#8594; wind proxy</text>
      <rect x="490" y="15" width="200" height="90" rx="10" fill="#FBF9F4" stroke="#C77E1F" stroke-width="1.4"/>
      <text x="590" y="40" text-anchor="middle" font-size="20" font-weight="700" fill="#B5402C">4</text>
      <text x="590" y="58" text-anchor="middle" font-size="11" font-weight="600" fill="#211E19">Winter smog</text>
      <text x="590" y="72" text-anchor="middle" font-size="9" fill="#5C574C">Dec 2025</text>
      <text x="590" y="94" text-anchor="middle" font-size="9" fill="#C77E1F" font-weight="700">blocked &#8594; no weather col</text>
    </g>
    <line x1="110" y1="105" x2="110" y2="130" stroke="#3E6E52" stroke-width="1.4" marker-end="url(#a2)"/>
    <line x1="350" y1="105" x2="350" y2="130" stroke="#3E6E52" stroke-width="1.4" marker-end="url(#a2)"/>
    <line x1="590" y1="105" x2="590" y2="130" stroke="#C77E1F" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#a3)"/>
    <rect x="10" y="135" width="440" height="40" rx="8" fill="rgba(62,110,82,.1)" stroke="#3E6E52" stroke-width="1.2"/>
    <text x="230" y="152" text-anchor="middle" font-size="10.5" font-weight="700" fill="#3E6E52" font-family="IBM Plex Mono, monospace">DAF-14 can add: festival-date flag + wind-speed proxy</text>
    <text x="230" y="168" text-anchor="middle" font-size="9" fill="#5C574C" font-family="IBM Plex Mono, monospace">catches Diwali + dust-storm misses</text>
    <rect x="470" y="135" width="220" height="40" rx="8" fill="rgba(199,126,31,.12)" stroke="#C77E1F" stroke-width="1.2" stroke-dasharray="4 3"/>
    <text x="580" y="152" text-anchor="middle" font-size="10.5" font-weight="700" fill="#C77E1F" font-family="IBM Plex Mono, monospace">DAF-14 / DAF-18 gap</text>
    <text x="580" y="168" text-anchor="middle" font-size="9" fill="#5C574C" font-family="IBM Plex Mono, monospace">no wind/weather in daily_17.parquet</text>
    <defs>
      <marker id="a2" viewBox="0 0 8 8" refX="6" refY="4" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#3E6E52"/></marker>
      <marker id="a3" viewBox="0 0 8 8" refX="6" refY="4" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#C77E1F"/></marker>
    </defs>
  </svg>
</div>
</div>